In [ ]:
# MarketMind Intelligence Platform V1
# Author: Sharique Mohammad
# Date: April 23, 2026

# MarketMind V1 - Market Data Analysis

**Notebook:** 02_market_data_analysis.ipynb  
**Purpose:** Analyze OHLCV market data with visualizations

---

## Overview

This notebook analyzes market data from `gold.ohlcv_bars`:
- Stock price trends (line charts)
- Volume analysis (bar charts)
- Price correlations (heatmap)
- Daily returns distribution
- Trading statistics

**Data Period:** Q1 2026 (January 2 - April 21)
**Tickers:** 20 S&P 500 stocks
**Records:** 1,240 daily bars

In [ ]:
# Import libraries
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Libraries imported successfully!')

## 1. Database Connection & Data Loading

In [ ]:
# PostgreSQL connection
DB_CONFIG = {
    'host': '172.31.32.1',
    'port': 5432,
    'database': 'marketmind_v1',
    'user': 'postgres',
    'password': '0940'
}

# Load OHLCV data
conn = psycopg2.connect(**DB_CONFIG)

query = """
SELECT 
    ticker,
    date,
    open,
    high,
    low,
    close,
    volume,
    vwap
FROM gold.ohlcv_bars
ORDER BY date, ticker
"""

df = pd.read_sql(query, conn)
conn.close()

# Convert date to datetime
df['date'] = pd.to_datetime(df['date'])

print(f'Loaded {len(df):,} records')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Tickers: {df["ticker"].nunique()}')
df.head()

## 2. Stock Price Trends - Line Charts

In [ ]:
# Select top 5 tickers by average volume for cleaner chart
top_tickers = df.groupby('ticker')['volume'].mean().nlargest(5).index.tolist()

# Plot closing prices
fig, ax = plt.subplots(figsize=(14, 7))

for ticker in top_tickers:
    ticker_data = df[df['ticker'] == ticker]
    ax.plot(ticker_data['date'], ticker_data['close'], marker='o', markersize=3, label=ticker, linewidth=2)

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Close Price ($)', fontsize=12, fontweight='bold')
ax.set_title('Stock Price Trends - Top 5 by Volume (Q1 2026)', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f'Top 5 tickers by volume: {top_tickers}')

## 3. Volume Analysis

In [ ]:
# Average volume by ticker
avg_volume = df.groupby('ticker')['volume'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(range(len(avg_volume)), avg_volume.values, color='steelblue', edgecolor='black', linewidth=1)

ax.set_xlabel('Ticker', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Daily Volume', fontsize=12, fontweight='bold')
ax.set_title('Average Trading Volume by Ticker (Q1 2026)', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(avg_volume)))
ax.set_xticklabels(avg_volume.index, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, avg_volume.values)):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{val/1e6:.1f}M',
            ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 4. Price Correlation Heatmap

In [ ]:
# Create pivot table of closing prices
price_pivot = df.pivot(index='date', columns='ticker', values='close')

# Calculate correlation matrix
corr_matrix = price_pivot.corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, 
            annot=True, 
            fmt='.2f', 
            cmap='RdYlGn', 
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={'label': 'Correlation'},
            ax=ax)

ax.set_title('Stock Price Correlation Matrix (Q1 2026)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print('Highest correlations (excluding self):')
corr_pairs = corr_matrix.unstack()
corr_pairs = corr_pairs[corr_pairs < 1.0]
print(corr_pairs.nlargest(5))

## 5. Daily Returns Distribution

In [ ]:
# Calculate daily returns for each ticker
df_sorted = df.sort_values(['ticker', 'date'])
df_sorted['daily_return'] = df_sorted.groupby('ticker')['close'].pct_change() * 100

# Remove NaN values
returns = df_sorted['daily_return'].dropna()

# Plot histogram
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(returns, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(returns.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {returns.mean():.2f}%')
ax.axvline(returns.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {returns.median():.2f}%')

ax.set_xlabel('Daily Return (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Daily Returns (All Tickers, Q1 2026)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Daily Returns Statistics:')
print(f'  Mean: {returns.mean():.2f}%')
print(f'  Median: {returns.median():.2f}%')
print(f'  Std Dev: {returns.std():.2f}%')
print(f'  Min: {returns.min():.2f}%')
print(f'  Max: {returns.max():.2f}%')

## 6. Single Stock Deep Dive - AAPL

In [ ]:
# Focus on AAPL
aapl = df[df['ticker'] == 'AAPL'].copy()

# Create figure with 2 subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot 1: OHLC as line chart with volume
ax1.plot(aapl['date'], aapl['high'], label='High', alpha=0.7, linewidth=1.5)
ax1.plot(aapl['date'], aapl['low'], label='Low', alpha=0.7, linewidth=1.5)
ax1.plot(aapl['date'], aapl['close'], label='Close', linewidth=2.5, color='darkblue')
ax1.fill_between(aapl['date'], aapl['low'], aapl['high'], alpha=0.2, color='gray')

ax1.set_ylabel('Price ($)', fontsize=12, fontweight='bold')
ax1.set_title('AAPL Stock Analysis (Q1 2026)', fontsize=14, fontweight='bold')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Plot 2: Volume
ax2.bar(aapl['date'], aapl['volume'], color='steelblue', alpha=0.6, edgecolor='black')
ax2.set_xlabel('Date', fontsize=12, fontweight='bold')
ax2.set_ylabel('Volume', fontsize=12, fontweight='bold')
ax2.set_title('Trading Volume', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f'AAPL Statistics (Q1 2026):')
print(f'  Opening Price (Jan 2): ${aapl.iloc[0]["open"]:.2f}')
print(f'  Closing Price (Apr 21): ${aapl.iloc[-1]["close"]:.2f}')
print(f'  Q1 Return: {((aapl.iloc[-1]["close"] / aapl.iloc[0]["open"]) - 1) * 100:.2f}%')
print(f'  Average Volume: {aapl["volume"].mean():,.0f}')

## 7. Top Gainers and Losers

In [ ]:
# Calculate Q1 returns for each ticker
q1_returns = []

for ticker in df['ticker'].unique():
    ticker_data = df[df['ticker'] == ticker].sort_values('date')
    if len(ticker_data) > 0:
        start_price = ticker_data.iloc[0]['open']
        end_price = ticker_data.iloc[-1]['close']
        q1_return = ((end_price / start_price) - 1) * 100
        q1_returns.append({'ticker': ticker, 'q1_return': q1_return})

returns_df = pd.DataFrame(q1_returns).sort_values('q1_return', ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))

colors = ['green' if x > 0 else 'red' for x in returns_df['q1_return']]
bars = ax.barh(range(len(returns_df)), returns_df['q1_return'], color=colors, edgecolor='black', linewidth=1)

ax.set_yticks(range(len(returns_df)))
ax.set_yticklabels(returns_df['ticker'])
ax.set_xlabel('Q1 2026 Return (%)', fontsize=12, fontweight='bold')
ax.set_title('Q1 2026 Performance - All Tickers', fontsize=14, fontweight='bold')
ax.axvline(0, color='black', linewidth=1)
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (bar, val) in enumerate(zip(bars, returns_df['q1_return'])):
    ax.text(val + (0.5 if val > 0 else -0.5), bar.get_y() + bar.get_height()/2.,
            f'{val:.1f}%',
            ha='left' if val > 0 else 'right', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print('Top 3 Gainers:')
print(returns_df.head(3))
print('\nTop 3 Losers:')
print(returns_df.tail(3))